# Lab 1 — Train a Decision Model to Play Pac-Man

Follow Kev's published **LoRA plus pointer-head** architecture. First train a fresh decision model from `Qwen/Qwen3.5-0.8B-Base` on the frozen `decision-v7` training suite. Then fine-tune your own adapter and head on Pac-Man labels. Original base matrices stay frozen, and LoRA changes the encoder's effective features. This is initial decision-model training over an already pretrained LLM.

**You train Pac-Man. Ghosts follow deterministic game code.** Start with the community browser game's maze and renderer, then use Kev to select the player's legal moves. This adaptation changes the original community demo, where Jev controls the ghosts.

0–10 compare architectures and play; 10–20 inspect initial training; 20–30 prepare player labels; **30–60 mandatory fine-tuning**; 60–80 compare; 80–90 debrief. Installation, downloads and the complete initial training run are prework, so we can keep the published recipe and the 90-minute class.

The target runtime is **Colab with one NVIDIA RTX PRO 6000 Blackwell GPU**. The full Server Edition has 96 GB VRAM. Check the actual allocation in the prework cell. The notebook uses BF16 inference and training autocast, with FP32 stored backbone, adapter and head parameters in the initial recipe. Colab does not guarantee this GPU, including on paid plans. Arrange access before class and record actual runtime cost. Both stages still need an instructor GPU preflight.

We reproduce the published **base stage**, then add our own Pac-Man domain adaptation. The released Kev-0.8B also has dates, documents/skills and calibration stages. This lab does not reproduce those stages or inherit the release's accuracy/calibration claims. [Recipe and compute](https://github.com/jaredpalmer/kev/blob/84847f0a883d900f7de5b7a57eaa341ca7f9a6b4/docs/model-cards/kev-0.8b.md#training-procedure).

## Game provenance

Game source: [codaaiteam/jev-pacman](https://github.com/codaaiteam/jev-pacman), MIT, pinned `8446fe74690cd61909bda91acfadbccb0f02b422`. Model source: [jaredpalmer/kev](https://github.com/jaredpalmer/kev). The model receives structured state, not screenshots. No ROM or paid API key is required for the notebook path.

## Prework: prepare the runtime

Save your own copy of this notebook in Colab. Open **Runtime > Change runtime type**, select the RTX PRO 6000 Blackwell option if your account offers it, then connect. Run the prework cells before class. Confirm the GPU name rather than relying on a menu label. A different allocation requires an instructor-approved, timed fallback.

Setup creates a separate Python 3.13 environment using Kev's locked dependencies, including PyTorch 2.8.0 with CUDA 12.8. It checks GPU identity, BF16 support and a CUDA forward/backward pass in that environment, then writes `runtime-preflight.json`. The notebook kernel only runs the teaching helpers and Colab callback. Complete installation and model downloads before class.

The target GPU is an assumption for this session, not a free-compute promise. [Colab availability](https://research.google.com/colaboratory/faq.html), [NVIDIA GPU specifications](https://www.nvidia.com/en-us/data-center/rtx-pro-6000-blackwell-server-edition/), [PyTorch Blackwell support](https://pytorch.org/blog/pytorch-2-7/).

In [ ]:
from pathlib import Path
import sys, json, os
LAB_DIR = Path.cwd() / 'pacman-kev-lab'
LAB_DIR.mkdir(exist_ok=True)
BUNDLED_FILES = {'api_client.py': '"""Call the notebook-local Kev server and validate decision probabilities."""\nimport json\nimport math\nimport os\nimport time\nfrom urllib.error import HTTPError, URLError\nfrom urllib.request import Request, urlopen\n\n\ndef call(path, body=None):\n    base = os.environ.get("KEV_BASE_URL", "http://127.0.0.1:8009").rstrip("/")\n    headers = {"Content-Type": "application/json"}\n    if os.environ.get("KEV_API_KEY"):\n        headers["Authorization"] = "Bearer " + os.environ["KEV_API_KEY"]\n    data = None if body is None else json.dumps(body).encode("utf-8")\n    started = time.perf_counter()\n    try:\n        with urlopen(Request(base + path, data=data, headers=headers), timeout=120) as resp:\n            value = json.load(resp)\n    except HTTPError as exc:\n        raise RuntimeError(f"Kev returned HTTP {exc.code}. Check credentials, request schema and the server log.") from None\n    except (URLError, TimeoutError) as exc:\n        raise RuntimeError("Cannot reach Kev. Check KEV_BASE_URL and whether the notebook-local server is running.") from exc\n    return value, (time.perf_counter() - started) * 1000\n\n\ndef distribution(answer, keys):\n    """Check API output before scoring it. API rounds to four decimal places."""\n    p = answer.get("probabilities", {})\n    if set(p) != set(keys) or any(not isinstance(v, (float, int)) or not math.isfinite(v) or not 0 <= v <= 1 for v in p.values()):\n        raise ValueError("Response probability keys or values do not match the question.")\n    if not math.isclose(sum(p.values()), 1, abs_tol=0.02):\n        raise ValueError("Response probabilities do not sum to approximately one.")\n    if answer.get("choice") not in p or p[answer["choice"]] < max(p.values()) - 1e-4:\n        raise ValueError("Response choice is not a highest-probability option.")\n    # Undo serialization rounding for metrics only. Keep raw responses in artifacts.\n    return {k: v / sum(p.values()) for k, v in p.items()}\n', 'pacman_lab.py': '"""Kev controls Pac-Man; deterministic game code controls both ghosts."""\nfrom collections import deque\nimport copy\nimport json\nfrom pathlib import Path\nimport random\nimport re\nimport time\n\nfrom api_client import call, distribution\n\nPACMAN_REVISION = "8446fe74690cd61909bda91acfadbccb0f02b422"\nPOLICY = "Control Pac-Man to collect dots and avoid ghosts. First avoid being caught this turn, including the programmed ghost move on even-numbered turns. Among safe moves, minimize maze distance to a remaining dot without crossing a ghost\'s current tile. Then prefer greater distance from the nearest ghost. Break remaining ties in order: up, down, left, right."\nDIRECTIONS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}\nOPPOSITE = {"up": "down", "down": "up", "left": "right", "right": "left"}\n\n\ndef maze_from_html(html):\n    match = re.search(r"const MAZE = (\\[.*?\\]);", html, re.S)\n    if not match:\n        raise ValueError("Pinned game maze was not found")\n    return json.loads(re.sub(r",\\s*]", "]", match.group(1)))\n\n\ndef position(actor):\n    return actor["row"], actor["column"]\n\n\ndef actor(pos, heading=None):\n    return {"row": pos[0], "column": pos[1], "heading": heading}\n\n\ndef destination(pos, direction):\n    dr, dc = DIRECTIONS[direction]\n    return pos[0] + dr, pos[1] + dc\n\n\ndef legal(maze, pos):\n    options = []\n    for direction in DIRECTIONS:\n        r, c = destination(pos, direction)\n        if 0 <= r < len(maze) and 0 <= c < len(maze[0]) and maze[r][c] != "#":\n            options.append(direction)\n    return options\n\n\ndef distances(maze, target, blocked=()):\n    found, queue = {target: 0}, deque([target])\n    while queue:\n        tile = queue.popleft()\n        for direction in legal(maze, tile):\n            nxt = destination(tile, direction)\n            if nxt not in found and nxt not in blocked:\n                found[nxt] = found[tile] + 1\n                queue.append(nxt)\n    return found\n\n\ndef ghost_step(maze, ghosts, target):\n    """Community straight-ahead/Manhattan chase, with fixed ties (no RNG)."""\n    result = []\n    for ghost in ghosts:\n        pos, heading = position(ghost), ghost.get("heading")\n        options = legal(maze, pos)\n        if heading in options:\n            move = heading\n        else:\n            choices = [d for d in options if d != OPPOSITE.get(heading)] or options\n            def manhattan(d):\n                nxt = destination(pos, d)\n                return abs(nxt[0] - target[0]) + abs(nxt[1] - target[1])\n            move = min(choices, key=manhattan)\n        result.append(actor(destination(pos, move), move))\n    return result\n\n\ndef initial_state(maze):\n    pac = next((r, c) for r, row in enumerate(maze) for c, ch in enumerate(row) if ch == "P")\n    ghosts = [actor((r, c)) for r, row in enumerate(maze) for c, ch in enumerate(row) if ch == "G"]\n    excluded = {pac, *(position(g) for g in ghosts)}\n    return {"maze": maze, "player": actor(pac), "ghosts": ghosts, "dots": [list(tile) for tile in sorted(distances(maze, pac)) if tile not in excluded], "turn": 0, "ghost_move_every": 2, "objective": "Collect every dot without being caught. Ghosts are controlled by deterministic game code."}\n\n\ndef body(state):\n    criteria = {d: f"Move {d} to row {destination(position(state[\'player\']), d)[0]}, column {destination(position(state[\'player\']), d)[1]}." for d in legal(state["maze"], position(state["player"]))}\n    return {"state": state, "model": "kev-latest", "questions": {"move": {"type": "choice", "instructions": POLICY, "criteria": criteria}}}\n\n\ndef transition(state, direction):\n    """One player turn, then a ghost turn every second player turn."""\n    if direction not in legal(state["maze"], position(state["player"])):\n        raise ValueError("Player action crosses a wall")\n    nxt = copy.deepcopy(state)\n    pac = destination(position(state["player"]), direction)\n    nxt["player"] = actor(pac, direction)\n    nxt["turn"] += 1\n    nxt["dots"] = [dot for dot in nxt["dots"] if tuple(dot) != pac]\n    caught = pac in [position(g) for g in nxt["ghosts"]]\n    if not nxt["dots"]:\n        return nxt, "win"  # community game checks the final dot first\n    if not caught and nxt["turn"] % nxt["ghost_move_every"] == 0:\n        nxt["ghosts"] = ghost_step(state["maze"], nxt["ghosts"], pac)\n        caught = pac in [position(g) for g in nxt["ghosts"]]\n    return nxt, "caught" if caught else None\n\n\ndef teacher(state):\n    """A stated heuristic for supervised imitation, not optimal Pac-Man play."""\n    current_ghosts = [position(g) for g in state["ghosts"]]\n    def rank(direction):\n        nxt, outcome = transition(state, direction)\n        if outcome == "win":\n            return (-1, 0, 0)\n        pac = position(nxt["player"])\n        routes = distances(state["maze"], pac, current_ghosts)\n        dot_distance = min((routes.get(tuple(dot), 999) for dot in state["dots"]), default=0)\n        separation = min((distances(state["maze"], pac).get(position(g), 999) for g in nxt["ghosts"]), default=999)\n        return (int(outcome == "caught"), dot_distance, -separation)\n    return min(legal(state["maze"], position(state["player"])), key=rank)\n\n\ndef make_data(html, directory, counts=(64, 16, 16), seed=7):\n    """Synthetic positions on the community maze, not human recordings."""\n    maze = maze_from_html(html)\n    initial = initial_state(maze)\n    tiles = list(distances(maze, position(initial["player"])))\n    rng, used, rows = random.Random(seed), set(), []\n    while len(rows) < sum(counts):\n        pac, ghost1, ghost2 = rng.sample(tiles, 3)\n        group = (pac, ghost1, ghost2)\n        if group in used or len(legal(maze, pac)) < 2:\n            continue\n        used.add(group)\n        state = copy.deepcopy(initial)\n        state["player"] = actor(pac)\n        state["ghosts"] = [actor(ghost1), actor(ghost2)]\n        state["dots"] = [list(t) for t in sorted(tiles) if t not in group and rng.random() < 0.3]\n        if not state["dots"]:\n            continue\n        state["turn"] = rng.randrange(2)\n        request = body(state)\n        request["questions"]["move"].update(label=teacher(state), src="pacman_teacher")\n        request.pop("model")\n        request["_meta"] = {"id": f"board-{len(rows):03d}", "group_id": f"positions-{group}", "source": "pacman_synthetic", "variant": "clean", "label_source": "deterministic safety/shortest-dot heuristic"}\n        rows.append(request)\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    offset, groups, labels = 0, {}, {}\n    for name, count in zip(["train", "development", "evaluation"], counts):\n        part = rows[offset:offset+count]\n        offset += count\n        (directory / f"pacman-{name}.jsonl").write_text("".join(json.dumps(row, ensure_ascii=False) + "\\n" for row in part), encoding="utf-8")\n        groups[name] = [row["_meta"]["group_id"] for row in part]\n        labels[name] = {d: sum(row["questions"]["move"]["label"] == d for row in part) for d in DIRECTIONS}\n    manifest = {"counts": dict(zip(["train", "development", "evaluation"], counts)), "labels": labels, "seed": seed, "game_commit": PACMAN_REVISION, "label_policy": POLICY, "groups": groups, "interpretation": "Authored synthetic positions on one maze. Measures imitation of a teacher, not human-level play or generalization to new mazes."}\n    (directory / "pacman-manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n", encoding="utf-8")\n    return manifest\n\n\ndef inference_request(record):\n    return {"state": record["state"], "model": "kev-latest", "questions": {key: {field: q[field] for field in ["type", "instructions", "criteria"]} for key, q in record["questions"].items()}}\n\n\ndef evaluate(path, predict=None):\n    if predict is None:\n        predict = lambda request: call("/v1/systemone", request)[0]\n    results = []\n    for line in Path(path).read_text(encoding="utf-8").splitlines():\n        record = json.loads(line)\n        started = time.perf_counter()\n        response = predict(inference_request(record))\n        answer = response["answers"]["move"]\n        distribution(answer, record["questions"]["move"]["criteria"])\n        _, outcome = transition(record["state"], answer["choice"])\n        results.append({"id": record["_meta"]["id"], "gold": record["questions"]["move"]["label"], "prediction": answer["choice"], "probabilities": answer["probabilities"], "caught_next_turn": outcome == "caught", "http_ms": (time.perf_counter() - started) * 1000})\n    if not results:\n        raise ValueError("No labelled decisions to evaluate")\n    return {"n": len(results), "accuracy": sum(row["gold"] == row["prediction"] for row in results) / len(results), "caught_next_turn": sum(row["caught_next_turn"] for row in results), "rows": results}\n\n\ndef rollout(maze, predict=None, max_turns=24):\n    if predict is None:\n        predict = lambda request: call("/v1/systemone", request)[0]\n    state = initial_state(maze)\n    initial_dots, frames, outcome = len(state["dots"]), [copy.deepcopy(state)], None\n    for _ in range(max_turns):\n        request = body(state)\n        answer = predict(request)["answers"]["move"]\n        distribution(answer, request["questions"]["move"]["criteria"])\n        state, outcome = transition(state, answer["choice"])\n        frames.append(copy.deepcopy(state))\n        if outcome:\n            break\n    return {"turns": state["turn"], "dots_collected": initial_dots - len(state["dots"]), "outcome": outcome or "turn_limit", "frames": frames}\n\n\nclass NotebookBridge:\n    def __init__(self, trace_path):\n        self.trace_path = Path(trace_path)\n        self.trace_path.parent.mkdir(parents=True, exist_ok=True)\n\n    def decide(self, state):\n        request = body(state)\n        response, elapsed = call("/v1/systemone", request)\n        distribution(response["answers"]["move"], request["questions"]["move"]["criteria"])\n        with self.trace_path.open("a", encoding="utf-8") as handle:\n            handle.write(json.dumps({"request": request, "response": response, "http_ms": elapsed}, ensure_ascii=False) + "\\n")\n        return response\n\n\ndef notebook_game(source_html, controller_js, license_text):\n    """Reuse the pinned community maze and renderer, replace its controller."""\n    html = source_html\n    html = re.sub(r\'<meta name="description"[^>]*>\', \'<meta name="description" content="Train Kev to control Pac-Man. Collect dots and avoid deterministic ghosts in this community browser game adaptation." />\', html, count=1)\n    def replace(pattern, replacement):\n        nonlocal html\n        html, count = re.subn(pattern, lambda _: replacement, html, count=1, flags=re.S)\n        if count != 1:\n            raise ValueError("Game source no longer matches the pinned adapter")\n    replace(r\'<a class="live".*?</a>\', \'<p class="live">Pac-Man lab — train the player</p>\')\n    replace(r\'<h1>.*?</h1>\', \'<h1>Kev plays Pac-Man</h1>\')\n    replace(r\'<p class="lede">.*?</p>\', \'<p class="lede">Collect dots and avoid the ghosts. Try a turn yourself, then let Kev control Pac-Man. Ghosts follow fixed game code.</p>\')\n    replace(r\'<details class="set".*?</details>\', \'<p><label>Player: <select id="lab-mode"><option value="human">Human — arrow keys / WASD</option><option value="kev">Kev — model decisions</option></select></label></p>\')\n    replace(r\'<footer>.*?</footer>\', \'<footer>Adapted from <a href="https://github.com/codaaiteam/jev-pacman">codaaiteam/jev-pacman</a> (MIT). The model controls Pac-Man; ghosts use deterministic game code.</footer>\')\n    replace(r\'const DEFAULT_BASE = .*?// ---------------------------------------------------------------------------\\n// Game\', \'// ---------------------------------------------------------------------------\\n// Game\')\n    replace(r\'const JEV_COST_PER_MTOK = .*?;\', \'const STEP_MS=600;\')\n    replace(r\'let startAt,\', \'let turn=0,gameGeneration=0,playerInfo=null,playerBusy=false;\\nlet startAt,\')\n    replace(r\'function resetState\\(\\)\\{\', \'function resetState(){\\n  turn=0;gameGeneration++;playerInfo=null;playerBusy=false;\')\n    replace(r\'const rows=\\[0,1\\].map.*?document.getElementById\\("hunt-rows"\\).innerHTML=rows;\', \'document.getElementById("hunt-rows").textContent=playerInfo ? `Pac-Man → ${playerInfo.choice}; probability ${playerInfo.probability.toFixed(2)}` : "Waiting for a player decision";\')\n    replace(r\'async function askJev\\(\\).*?function startLoop\\(\\)\', controller_js + \'\\nfunction startLoop()\')\n    html = html.replace(\'function pause(){running=false;\', \'function pause(){gameGeneration++;playerBusy=false;running=false;\')\n    html = html.replace(\'Math.random()*0.3\', \'0\')\n    html = html.replace("Jev\'s ghosts are hunting — one API call picks both directions", \'Pac-Man decisions — ghosts follow fixed code\')\n    html = html.replace(\' · $${cost.toFixed(5)}\', \'\')\n    html = re.sub(r\'<span>· <b>\\$\\$\\{cost\\.toFixed\\(5\\)\\}</b></span>\', \'\', html)\n    html = html.replace(\'Jev calls\', \'model calls\').replace(\'Jev decisions\', \'model decisions\')\n    html = html.replace("Run! Jev\'s ghosts are hunting you", \'Collect dots and avoid ghosts\')\n    html = html.replace(\'Jev caught you\', \'The ghosts caught Pac-Man\').replace(\'You outran Jev.\', \'Maze cleared.\').replace("outrun Jev\'s ghosts", \'collect dots and avoid ghosts\')\n    html = html.replace(\'<title>Jev Pac-Man — you drive, the AI ghosts hunt you</title>\', \'<title>Kev plays Pac-Man</title>\')\n    html += \'\\n<!--\\n\' + license_text + \'\\n-->\\n\'\n    return html\n', 'cloud_runtime.py': '"""Colab RTX PRO 6000 Blackwell helpers for an interactive Kev notebook.\n\nSetup runs before the 90-minute session. GPU training duration needs a preflight.\n"""\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom urllib.error import URLError\nfrom urllib.request import urlopen\n\nCODE_REVISION = "84847f0a883d900f7de5b7a57eaa341ca7f9a6b4"\nMODEL_RUN = "jaredpalmer/kev-0.8b@bf75a6a8848ea6960ff2ed108d9ed44c2941174f"\nTARGET_GPU = "RTX PRO 6000 Blackwell"\nBASE_MODEL = "Qwen/Qwen3.5-0.8B-Base"\nBASE_REVISION = "dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68"\nTRAINING_SUITE = "evals/v7/decision-v7"\n\nGPU_PREFLIGHT = """\nimport json, torch\nif not torch.cuda.is_available():\n    raise RuntimeError(\'Select a GPU in Runtime > Change runtime type, then reconnect.\')\nprops = torch.cuda.get_device_properties(0)\nbf16 = torch.cuda.is_bf16_supported()\n# Exercise CUDA kernels and a backward pass in the installed Kev environment.\ndtype = torch.bfloat16 if bf16 else torch.float32\nx = torch.randn(32, 32, device=\'cuda\', dtype=dtype, requires_grad=True)\n(x @ x.T).float().square().mean().backward()\ntorch.cuda.synchronize()\nprint(json.dumps({\'name\': props.name, \'memory_gib\': props.total_memory / 2**30,\n                  \'compute_capability\': [props.major, props.minor],\n                  \'torch\': torch.__version__, \'cuda\': torch.version.cuda,\n                  \'bf16_supported\': bf16, \'kernel_backward\': \'passed\'}))\n"""\n\n\ndef validate_gpu(info, allow_other_gpu=False):\n    """Select precision only after checking the classroom hardware/software contract."""\n    target = TARGET_GPU.lower() in info["name"].lower()\n    if not target and not allow_other_gpu:\n        raise RuntimeError(f"This lab targets {TARGET_GPU}; received {info[\'name\']}. "\n                           "Select the target GPU if available, or use the instructor\'s validated fallback.")\n    # The pinned PyTorch 2.8 CUDA 12.8 wheel does not support Pascal/P100.\n    if tuple(info["compute_capability"]) < (7, 0):\n        raise RuntimeError("This environment does not support P100/Pascal GPUs. Use the target GPU or a T4 fallback.")\n    if not info["torch"].startswith("2.8.0") or tuple(map(int, info["cuda"].split(".")[:2])) < (12, 8):\n        raise RuntimeError("The lab requires its pinned PyTorch 2.8.0 / CUDA 12.8 environment. Re-run setup.")\n    if info["kernel_backward"] != "passed":\n        raise RuntimeError("CUDA forward/backward preflight did not pass.")\n    if target and not info["bf16_supported"]:\n        raise RuntimeError("The target GPU must support BF16. Check the installed CUDA environment.")\n    return "bf16" if info["bf16_supported"] else "fp32"\n\n\nclass CloudRuntime:\n    def __init__(self, workspace, allow_other_gpu=False):\n        self.workspace = Path(workspace).resolve()\n        self.repo = self.workspace / "kev"\n        self.python = self.repo / ".venv" / "bin" / "python"\n        self.process = None\n        self.log = None\n        self.allow_other_gpu = allow_other_gpu\n        self.gpu = None\n        self.dtype = "bf16"\n\n    def setup(self):\n        self.workspace.mkdir(parents=True, exist_ok=True)\n        subprocess.run(["nvidia-smi"], check=True)\n        if not shutil.which("uv"):\n            subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)\n        if not self.repo.exists():\n            subprocess.run(["git", "clone", "https://github.com/jaredpalmer/kev.git", str(self.repo)], check=True)\n            subprocess.run(["git", "checkout", CODE_REVISION], cwd=self.repo, check=True)\n        actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=self.repo, text=True).strip()\n        if actual != CODE_REVISION:\n            raise RuntimeError("Existing Kev checkout differs from the lab pin. Use a fresh workspace.")\n        subprocess.run(["uv", "sync", "--frozen", "--no-dev", "--extra", "serve", "--python", "3.13"], cwd=self.repo, check=True)\n        self.gpu = json.loads(subprocess.check_output([str(self.python), "-c", GPU_PREFLIGHT], cwd=self.repo, text=True))\n        self.dtype = validate_gpu(self.gpu, self.allow_other_gpu)\n        (self.workspace / "runtime-preflight.json").write_text(json.dumps(self.gpu, indent=2) + "\\n", encoding="utf-8")\n        print(json.dumps(self.gpu, indent=2))\n        print(f"Prepared pinned Kev environment using {self.dtype}. Download weights before class.")\n\n    def environment(self):\n        env = dict(os.environ)\n        # BF16 on the target GPU. A validated T4 fallback uses FP32.\n        env.update(KEV_DTYPE=self.dtype, KEV_BACKEND="torch", KEV_FUSED="0", KEV_CUDA_GRAPHS="0", PYTHONUNBUFFERED="1")\n        env.pop("KEV_API_KEY", None)  # this server binds only to notebook-local loopback\n        return env\n\n    def prepare_training(self):\n        """Fetch verified training data/base weights and inspect exactly what can train."""\n        audit_path = self.workspace / "trainable-parameters.json"\n        code = """\nimport json, sys\nfrom kev.suite import load_split\nfrom kev.model import DecisionModel, load_tokenizer\nrows = load_split(sys.argv[1], \'train\')\ntok = load_tokenizer(sys.argv[2], revision=sys.argv[3])\nmodel = DecisionModel(sys.argv[2], tok, \'cpu\', lora=16, revision=sys.argv[3], lora_targets=\'all\')\ntrainable = {n:p.numel() for n,p in model.named_parameters() if p.requires_grad}\nassert trainable and any(n.startswith(\'head.\') for n in trainable)\nassert all(n.startswith(\'head.\') or \'lora_A\' in n or \'lora_B\' in n for n in trainable)\nassert any(\'lora_A\' in n for n in trainable)\nreport = {\'base\':sys.argv[2], \'base_revision\':sys.argv[3], \'suite_records\':len(rows),\n          \'trainable_parameters\':trainable, \'original_base_matrices_frozen\':True,\n          \'effective_encoder_features_fixed\':False, \'lora_rank\':16, \'lora_alpha\':32}\nwith open(sys.argv[4], \'w\') as out: json.dump(report, out, indent=2)\nprint(\'Prepared\', len(rows), \'verified training records and base weights. LoRA/head audit saved.\')\n"""\n        subprocess.run([str(self.python), "-c", code, TRAINING_SUITE, BASE_MODEL, BASE_REVISION, str(audit_path)],\n                       cwd=self.repo, env=self.environment(), check=True)\n        return json.loads(audit_path.read_text())\n\n    def pretraining_command(self, output, steps=0):\n        if steps < 0:\n            raise ValueError("steps must be nonnegative; 0 runs the complete two-epoch recipe")\n        recipe = json.loads((self.repo / "experiments/q35-08b.json").read_text())[2]\n        assert recipe["base"] == BASE_MODEL and recipe["base_revision"] == BASE_REVISION\n        if self.dtype != "bf16":\n            raise RuntimeError("The published initial recipe requires a BF16-capable GPU.")\n        command = [str(self.python), "-m", "kev.train", "--suite", TRAINING_SUITE,\n                   "--out", str(Path(output).resolve()), "--device", "cuda",\n                   "--lora", "16", "--lora_targets", "all", "--head_dim", "256",\n                   "--weights_dtype", "fp32"]\n        for key, value in recipe.items():\n            command += ["--" + key, str(value)]\n        if steps:\n            command += ["--max_steps", str(steps)]\n        return command\n\n    def pretrain(self, output, steps=0, timeout_minutes=90):\n        """Published decision-v7 base stage, without --init_from. Full run is prework."""\n        return self._train(self.pretraining_command(output, steps), output, timeout_minutes, "initial")\n\n    def start(self, run=MODEL_RUN):\n        self.stop()\n        self.log = open(self.workspace / "server.log", "w", encoding="utf-8")\n        self.process = subprocess.Popen([str(self.python), "-m", "kev.serve", "--run", str(run), "--host", "127.0.0.1", "--port", "8009"], cwd=self.repo, env=self.environment(), stdout=self.log, stderr=subprocess.STDOUT)\n        os.environ["KEV_BASE_URL"] = "http://127.0.0.1:8009"\n        os.environ.pop("KEV_API_KEY", None)\n        deadline = time.monotonic() + 900\n        while time.monotonic() < deadline:\n            if self.process.poll() is not None:\n                raise RuntimeError("Kev stopped while loading. Inspect server.log in the notebook workspace.")\n            try:\n                with urlopen("http://127.0.0.1:8009/v1/models", timeout=3) as response:\n                    models = json.load(response)\n                (self.workspace / "models.json").write_text(json.dumps(models, indent=2) + "\\n", encoding="utf-8")\n                print("Kev ready at notebook-local localhost:8009")\n                return models\n            except (URLError, TimeoutError):\n                time.sleep(3)\n        self.stop()\n        raise TimeoutError("Model startup exceeded 15 minutes. Inspect server.log and connection status.")\n\n    def stop(self):\n        if self.process is not None and self.process.poll() is None:\n            self.process.terminate()\n            try:\n                self.process.wait(timeout=30)\n            except subprocess.TimeoutExpired:\n                self.process.kill()\n                self.process.wait()\n        self.process = None\n        if self.log is not None:\n            self.log.close()\n            self.log = None\n\n    def training_command(self, training_data, output, init_from, steps=0, max_state=2048):\n        # Read architecture from the checkpoint rather than guessing compatible flags.\n        code = "import json,sys; from kev.checkpoint import Checkpoint; m=Checkpoint(sys.argv[1]).meta; print(json.dumps({k:getattr(m,k) for k in [\'base\',\'base_revision\',\'lora\',\'head_dim\',\'option_isolation\',\'special_embeddings\',\'weights_dtype\',\'weights\']}))"\n        meta = json.loads(subprocess.check_output([str(self.python), "-c", code, str(init_from)], cwd=self.repo, env=self.environment(), text=True))\n        if meta["weights"] != "lora":\n            raise RuntimeError("This classroom exercise expects the pinned small LoRA checkpoint.")\n        command = [str(self.python), "-m", "kev.train", "--init_from", str(init_from), "--data", str(Path(training_data).resolve()), "--out", str(Path(output).resolve()), "--base", meta["base"], "--lora", str(meta["lora"]), "--head_dim", str(meta["head_dim"]), "--option_isolation", str(int(meta["option_isolation"])), "--special_embeddings", str(int(meta["special_embeddings"])), "--weights_dtype", meta["weights_dtype"], "--dtype", self.dtype, "--device", "cuda", "--epochs", "2", "--lr", "2e-5", "--batch", "1", "--accum", "8", "--max_steps", str(steps), "--max_state", str(max_state), "--checkpointing", "1", "--p_none", "0", "--p_none_distract", "0", "--p_distract", "0", "--seed", "7"]\n        command[command.index("--max_state") + 1] = str(max_state)\n        command[command.index("--dtype") + 1] = self.dtype\n        # The pinned release targets all modules (including hybrid projections).\n        # warm_start checks that its adapter tensors match the training model.\n        command += ["--lora_targets", "all"]\n        if meta["base_revision"]:\n            command += ["--base_revision", meta["base_revision"]]\n        return command\n\n    def finetune(self, training_data, output, init_from, steps=0):\n        command = self.training_command(training_data, output, init_from, steps)\n        return self._train(command, output, 20, "pacman")\n\n    def _train(self, command, output, timeout_minutes, stage):\n        output = Path(output).resolve()\n        if output.exists():\n            raise RuntimeError("Choose a new checkpoint output directory for each training run.")\n        self.stop()  # release the inference model\'s GPU allocation before training\n        started = time.perf_counter()\n        try:\n            subprocess.run(command, cwd=self.repo, env=self.environment(), check=True, timeout=timeout_minutes * 60)\n        except subprocess.TimeoutExpired:\n            raise RuntimeError(f"Training exceeded the {timeout_minutes}-minute run budget. Keep the log and use the instructor\'s prepared checkpoint for comparison.") from None\n        evidence = {"stage": stage, "command": command, "elapsed_seconds_including_load_save": time.perf_counter() - started,\n                    "gpu": self.gpu, "code_revision": CODE_REVISION}\n        (output / "run-evidence.json").write_text(json.dumps(evidence, indent=2) + "\\n", encoding="utf-8")\n        metrics = json.loads((output / "training_metrics.json").read_text())\n        if metrics["optimizer_steps"] < 1:\n            raise RuntimeError("Training completed without a positive optimizer step.")\n        print(f"{stage} training finished in {evidence[\'elapsed_seconds_including_load_save\'] / 60:.1f} minutes")\n        return output\n', 'games/player-controller.js': "// Course adaptation: the model controls Pac-Man; ghosts use fixed code.\nfunction observation(){\n  return {\n    maze: MAZE,\n    player: {row:pac.r,column:pac.c,heading:dir?dir.name:null},\n    ghosts: ghosts.map(g=>({row:g.r,column:g.c,heading:g.dir?g.dir.name:null})),\n    dots: [...dots].map(k=>k.split(',').map(Number)).sort((a,b)=>a[0]-b[0]||a[1]-b[1]),\n    turn, ghost_move_every:2,\n    objective:'Collect every dot without being caught. Ghosts are controlled by deterministic game code.'\n  };\n}\nfunction stepGhosts(){\n  ghosts=ghosts.map(g=>{\n    const legal=legalDirs(g);\n    let move=g.dir&&legal.some(d=>d.name===g.dir.name)?g.dir:null;\n    if(!move){\n      const nonReverse=legal.filter(d=>d.name!==OPP[g.dir&&g.dir.name]);\n      move=greedyToward(g,nonReverse.length?nonReverse:legal,pac);\n    }\n    return {r:g.r+move.dr,c:g.c+move.dc,dir:move};\n  });\n}\nfunction finish(kind){running=false;over=kind;stopLoop();render();}\nasync function tick(){\n  if(!running||playerBusy)return;\n  const generation=gameGeneration;\n  let move=null;\n  const mode=document.getElementById('lab-mode').value;\n  if(mode==='kev'){\n    playerBusy=true;\n    const t=performance.now();\n    try{\n      const result=await google.colab.kernel.invokeFunction('pacman.decide',[observation()],{});\n      if(generation!==gameGeneration)return;\n      const data=result.data['application/json'];\n      const ans=data&&data.answers&&data.answers.move;\n      if(!ans||!legalDirs(pac).some(d=>d.name===ans.choice))throw new Error('The model did not return a legal player move.');\n      move=DIR_BY_NAME[ans.choice];\n      playerInfo={choice:ans.choice,probability:ans.probabilities[ans.choice]};\n      latency=Math.round(performance.now()-t);decisions++;err=null;\n    }catch(e){\n      if(generation!==gameGeneration)return;\n      playerBusy=false;running=false;stopLoop();err=String(e.message||e);render();return;\n    }\n    playerBusy=false;\n  }else{\n    if(desiredDir&&!isWall(pac.r+desiredDir.dr,pac.c+desiredDir.dc))move=desiredDir;\n    else if(dir&&!isWall(pac.r+dir.dr,pac.c+dir.dc))move=dir;\n    if(!move)return;\n    playerInfo=null;\n  }\n  pac={r:pac.r+move.dr,c:pac.c+move.dc};dir=move;turn++;\n  dots.delete(kkey(pac.r,pac.c));\n  if(dots.size===0){finish('win');return;}\n  if(ghosts.some(g=>g.r===pac.r&&g.c===pac.c)){finish('caught');return;}\n  if(turn%2===0)stepGhosts();\n  if(ghosts.some(g=>g.r===pac.r&&g.c===pac.c)){finish('caught');return;}\n  render();\n}\n", 'community/index.html': '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="utf-8" />\n<meta name="viewport" content="width=device-width, initial-scale=1" />\n<title>Jev Pac-Man — you drive, the AI ghosts hunt you</title>\n<meta name="description" content="You drive Pac-Man; Jev, TypeSafe AI\'s decision model, drives the ghosts hunting you — one real API call picks both directions each turn. A single-file open-source game on the Jev decision API." />\n<style>\n  :root{\n    --bg:#0b0d13; --panel:#12141c; --panel2:#171a24; --border:#242838; --border2:#2f3446;\n    --text:#e9ebf2; --muted:#8b90a6; --accent:#7c8cff; --rose:#ff6b6b; --cyan:#6be0ff;\n    --amber:#edc22e; --lime:#7ff0a8; --mono:ui-monospace,SFMono-Regular,Menlo,monospace;\n    --radius:14px;\n  }\n  *{box-sizing:border-box}\n  body{margin:0;background:var(--bg);color:var(--text);font:15px/1.5 -apple-system,system-ui,Segoe UI,Roboto,sans-serif;\n    background-image:radial-gradient(1200px 600px at 70% -10%,rgba(124,140,255,.10),transparent);}\n  .wrap{max-width:520px;margin:0 auto;padding:26px 16px 40px}\n  a{color:var(--accent);text-decoration:none} a:hover{text-decoration:underline}\n  h1{font-size:26px;letter-spacing:-.02em;margin:0 0 4px}\n  .lede{color:var(--muted);margin:0 0 16px;font-size:14.5px}\n  .live{display:inline-flex;align-items:center;gap:6px;font-weight:600;font-size:13.5px;\n    background:rgba(124,140,255,.12);border:1px solid rgba(124,140,255,.4);border-radius:999px;padding:4px 12px;margin-bottom:16px}\n  .set{border:1px solid var(--border);background:var(--panel);border-radius:12px;padding:12px 14px;margin-bottom:16px}\n  .set summary{cursor:pointer;font-size:13.5px;color:var(--muted);list-style:none}\n  .set summary::-webkit-details-marker{display:none}\n  .set summary b{color:var(--text)}\n  .set-body{margin-top:12px;display:flex;flex-direction:column;gap:10px}\n  .set label{font-size:12px;color:var(--muted);display:block;margin-bottom:4px}\n  .set input,.set select{width:100%;background:var(--bg);border:1px solid var(--border2);color:var(--text);\n    border-radius:8px;padding:8px 10px;font:13px var(--mono)}\n  .set .row{display:flex;gap:10px}.set .row>div{flex:1}\n  .set .save{align-self:flex-start;background:var(--accent);color:#0b0d13;border:none;border-radius:8px;\n    padding:8px 16px;font-weight:700;cursor:pointer;font-size:13px}\n  .set .hint{font-size:12px;color:var(--muted)}\n  .btn{border:1px solid var(--border2);background:var(--panel2);color:var(--text);border-radius:9px;\n    padding:9px 15px;font-size:14px;font-weight:600;cursor:pointer}\n  .btn:hover{border-color:var(--accent)}\n  .btn-primary{background:var(--accent);color:#0b0d13;border-color:var(--accent)}\n  .bar{display:flex;justify-content:space-between;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:12px}\n  .status{font-weight:600;font-size:15px}\n  .status.play{color:var(--accent)} .status.win{color:var(--lime)} .status.lose{color:var(--rose)}\n  .meta{font-size:12.5px;color:var(--muted);display:flex;gap:9px;flex-wrap:wrap;font-family:var(--mono)}\n  .meta b{color:var(--text)}\n  .hunt{background:var(--panel2);border:1px solid var(--border);border-radius:10px;padding:11px 14px;margin-bottom:14px}\n  .hunt-head{font-size:12.5px;color:var(--muted);margin-bottom:9px}\n  .hunt-rows{display:flex;flex-direction:column;gap:9px}\n  .hunt-row{display:flex;align-items:center;gap:12px}\n  .hunt-arrow{font-size:22px;line-height:1;width:26px;text-align:center;flex:none}\n  .hunt-arrow.g0{color:var(--rose)} .hunt-arrow.g1{color:var(--cyan)} .hunt-arrow.idle{color:var(--muted);opacity:.5}\n  .hunt-main{flex:1;min-width:0}\n  .hunt-top{display:flex;justify-content:space-between;gap:8px;font-size:12px;color:var(--muted);margin-bottom:5px}\n  .hunt-top b{color:var(--text)} .hunt-top .fb{color:var(--amber)}\n  .dot{width:9px;height:9px;border-radius:50%;display:inline-block;margin-right:6px}\n  .dot.g0{background:var(--rose)} .dot.g1{background:var(--cyan)}\n  .hunt-track{height:6px;border-radius:4px;background:var(--border);overflow:hidden}\n  .hunt-fill{height:100%;border-radius:4px;transition:width .25s ease}\n  .hunt-fill.g0{background:linear-gradient(90deg,var(--rose),#ff9f7f)} .hunt-fill.g1{background:linear-gradient(90deg,var(--cyan),#8fb4ff)}\n  .board-wrap{position:relative;max-width:420px;margin:0 auto}\n  #board{display:grid;gap:0;background:var(--bg);border-radius:8px;padding:6px;outline:none}\n  #board:focus-visible{box-shadow:0 0 0 2px var(--accent)}\n  .cell{position:relative;display:flex;align-items:center;justify-content:center}\n  .wall{background:#1b2030;border-radius:3px;box-shadow:inset 0 0 0 1px var(--border2)}\n  .pdot{width:18%;height:18%;border-radius:50%;background:var(--amber);opacity:.85}\n  .pac{width:76%;height:76%;border-radius:50%;background:var(--accent);box-shadow:0 0 9px var(--accent);transition:all 160ms linear}\n  .ghost{width:72%;height:72%;border-radius:50% 50% 28% 28%;transition:all 160ms linear;box-shadow:0 0 7px rgba(0,0,0,.4)}\n  .ghost.g0{background:var(--rose)} .ghost.g1{background:var(--cyan)}\n  .over{position:absolute;inset:0;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;\n    background:rgba(8,10,16,.85);border-radius:8px;text-align:center;padding:16px}\n  .over b{font-size:19px}.over.win b{color:var(--lime)}.over.lose b{color:var(--rose)}\n  .over span{font-size:13px;color:var(--muted)}\n  .hint{text-align:center;font-size:12.5px;color:var(--muted);margin-top:11px}\n  .hint kbd{font-family:var(--mono);background:var(--panel2);border:1px solid var(--border);border-radius:4px;padding:1px 6px;font-size:11.5px;color:var(--text)}\n  .dpad{display:grid;grid-template-columns:repeat(3,44px);grid-template-rows:repeat(3,44px);gap:6px;justify-content:center;margin:12px auto 0}\n  .dpad button{border:1px solid var(--border);background:var(--panel2);color:var(--text);border-radius:9px;font-size:18px;cursor:pointer}\n  .dpad button:active{background:var(--accent);color:#08101c}\n  .dpad .up{grid-area:1/2}.dpad .left{grid-area:2/1}.dpad .right{grid-area:2/3}.dpad .down{grid-area:3/2}\n  .actions{display:flex;gap:10px;justify-content:center;margin-top:14px;flex-wrap:wrap}\n  .err{margin-top:10px;text-align:center;color:var(--amber);font-size:13px}\n  footer{margin-top:28px;text-align:center;font-size:12.5px;color:var(--muted);line-height:1.8}\n</style>\n</head>\n<body>\n<div class="wrap">\n  <a class="live" href="https://jevtypesafeai.com/games/jev-pacman" target="_blank" rel="noopener">▶ Play live (no setup) on jevtypesafeai.com</a>\n  <h1>Jev Pac-Man</h1>\n  <p class="lede">You drive Pac-Man with the keyboard. <b>Jev</b> — TypeSafe AI\'s decision model — drives the ghosts, deciding how to hunt you in one real API call per turn. Eat every dot before they catch you.</p>\n\n  <details class="set" id="settings">\n    <summary>⚙︎ <b>Jev API key</b> — needed to run locally (or just <a href="https://jevtypesafeai.com/games/jev-pacman" target="_blank" rel="noopener">play live</a>)</summary>\n    <div class="set-body">\n      <div class="row">\n        <div><label>API key</label><input id="k-key" type="password" placeholder="jv_live_… (from jevtypesafeai.com) or ts_… (TypeSafe)" /></div>\n      </div>\n      <div><label>Endpoint</label>\n        <select id="k-base">\n          <option value="https://jevtypesafeai.com/api/v1/decide">jevtypesafeai.com — hosted (browser-ready)</option>\n          <option value="https://api.typesafe.ai/v1/systemone">api.typesafe.ai — official (may need a proxy: CORS)</option>\n        </select>\n      </div>\n      <button class="save" id="k-save">Save</button>\n      <p class="hint">Get a key free at <a href="https://jevtypesafeai.com" target="_blank" rel="noopener">jevtypesafeai.com</a>. Your key is stored only in this browser (localStorage) and sent straight to the endpoint above.</p>\n    </div>\n  </details>\n\n  <div class="bar">\n    <span class="status" id="status">You drive Pac-Man — outrun Jev\'s ghosts</span>\n    <span class="meta" id="meta"></span>\n  </div>\n\n  <div class="hunt">\n    <div class="hunt-head">Jev\'s ghosts are hunting — one API call picks both directions</div>\n    <div class="hunt-rows" id="hunt-rows"></div>\n  </div>\n\n  <div class="board-wrap">\n    <div id="board" tabindex="0" role="application" aria-label="Pac-Man board — arrow keys or WASD"></div>\n    <div class="over" id="over" style="display:none"></div>\n  </div>\n\n  <div class="hint">Use <kbd>← ↑ ↓ →</kbd> or <kbd>W A S D</kbd> to steer Pac-Man</div>\n\n  <div class="dpad" aria-hidden="true">\n    <button class="up" data-dir="up">↑</button>\n    <button class="left" data-dir="left">←</button>\n    <button class="right" data-dir="right">→</button>\n    <button class="down" data-dir="down">↓</button>\n  </div>\n\n  <div class="actions" id="actions"></div>\n  <div class="err" id="err" style="display:none"></div>\n\n  <footer>\n    Built on <a href="https://jevtypesafeai.com" target="_blank" rel="noopener"><b>Jev</b></a> — TypeSafe AI\'s System One decision model.<br/>\n    Every ghost move is a real typed decision. <a href="https://jevtypesafeai.com/games/jev-pacman" target="_blank" rel="noopener">Play live</a> · <a href="https://jevtypesafeai.com" target="_blank" rel="noopener">Get an API key</a> · <a href="https://github.com/codaaiteam/jev-pacman" target="_blank" rel="noopener">Source</a>\n  </footer>\n</div>\n\n<script>\n// ---------------------------------------------------------------------------\n// Jev connection — one typed decision per call. Works with a key from\n// jevtypesafeai.com (hosted, browser-ready) or the official TypeSafe API.\n// ---------------------------------------------------------------------------\nconst DEFAULT_BASE = "https://jevtypesafeai.com/api/v1/decide";\nconst JEV = {\n  get key(){ try { return localStorage.getItem("jev_key") || ""; } catch { return ""; } },\n  set key(v){ try { localStorage.setItem("jev_key", v); } catch {} },\n  get base(){ try { return localStorage.getItem("jev_base") || DEFAULT_BASE; } catch { return DEFAULT_BASE; } },\n  set base(v){ try { localStorage.setItem("jev_base", v); } catch {} },\n};\nasync function jevDecide(state, questions){\n  if(!JEV.key) { const e = new Error("Add your Jev API key in settings above (or play live)."); e.code = "nokey"; throw e; }\n  const res = await fetch(JEV.base, {\n    method:"POST",\n    headers:{ "Content-Type":"application/json", "Authorization":"Bearer "+JEV.key },\n    body: JSON.stringify({ model:"jev-latest", state, questions }),\n  });\n  if(res.status===401){ const e=new Error("Invalid or missing API key."); e.code=401; throw e; }\n  if(res.status===429){ const e=new Error("Rate limited — one moment."); e.code=429; throw e; }\n  const data = await res.json().catch(()=>({}));\n  if(!res.ok) throw new Error(data.error || ("HTTP "+res.status));\n  return data; // { answers:{k:{choice,probabilities,confidence}}, usage:{input_tokens} }\n}\n// settings UI\n(function(){\n  const key = document.getElementById("k-key"), base = document.getElementById("k-base");\n  key.value = JEV.key; base.value = JEV.base;\n  if(!JEV.key) document.getElementById("settings").open = true;\n  document.getElementById("k-save").onclick = () => {\n    JEV.key = key.value.trim(); JEV.base = base.value;\n    document.getElementById("settings").open = false;\n  };\n})();\n\n// ---------------------------------------------------------------------------\n// Game\n// ---------------------------------------------------------------------------\nconst JEV_COST_PER_MTOK = 0.042, STEP_MS = 160, THROTTLE_MS = 1600, RATE_WAIT_MS = 2500;\nconst MAZE = [\n  "#############","#...........#","#.##.###.##.#","#.#.......#.#","#.#.##.##.#.#",\n  "#...#.G.#...#","#.#.#.#.#.#.#","#.#...P...#.#","#.#.#.#.#.#.#","#...#.G.#...#",\n  "#.#.##.##.#.#","#.#.......#.#","#.##.###.##.#","#...........#","#############",\n];\nconst ROWS = MAZE.length, COLS = MAZE[0].length;\nconst DIRS4 = [{name:"up",dr:-1,dc:0},{name:"down",dr:1,dc:0},{name:"left",dr:0,dc:-1},{name:"right",dr:0,dc:1}];\nconst ARROW = {up:"↑",down:"↓",left:"←",right:"→"};\nconst OPP = {up:"down",down:"up",left:"right",right:"left"};\nconst DIR_BY_NAME = {up:DIRS4[0],down:DIRS4[1],left:DIRS4[2],right:DIRS4[3]};\nconst KEY_TO_DIR = {ArrowUp:"up",w:"up",W:"up",ArrowDown:"down",s:"down",S:"down",ArrowLeft:"left",a:"left",A:"left",ArrowRight:"right",d:"right",D:"right"};\n\nconst WALLS = MAZE.map(r=>r.split("").map(ch=>ch==="#"));\nlet PAC_START={r:7,c:6}; const GHOST_STARTS=[];\nfor(let r=0;r<ROWS;r++)for(let c=0;c<COLS;c++){const ch=MAZE[r][c];if(ch==="P")PAC_START={r,c};else if(ch==="G")GHOST_STARTS.push({r,c});}\nconst inB=(r,c)=>r>=0&&r<ROWS&&c>=0&&c<COLS;\nconst isWall=(r,c)=>!inB(r,c)||WALLS[r][c];\nconst kkey=(r,c)=>r+","+c;\nconst legalDirs=t=>DIRS4.filter(d=>!isWall(t.r+d.dr,t.c+d.dc));\nfunction initialDots(){\n  const skip=new Set([kkey(PAC_START.r,PAC_START.c),...GHOST_STARTS.map(g=>kkey(g.r,g.c))]);\n  const dots=new Set(),seen=new Set([kkey(PAC_START.r,PAC_START.c)]);let q=[PAC_START];\n  while(q.length){const nx=[];for(const cur of q)for(const d of DIRS4){const nr=cur.r+d.dr,nc=cur.c+d.dc;if(isWall(nr,nc))continue;const k=kkey(nr,nc);if(seen.has(k))continue;seen.add(k);if(!skip.has(k))dots.add(k);nx.push({r:nr,c:nc});}q=nx;}\n  return dots;\n}\nconst TOTAL_DOTS = initialDots().size;\nfunction greedyToward(from,options,target){\n  if(!options.length)return null;let best=options[0],bd=Infinity;\n  for(const d of options){const nr=from.r+d.dr,nc=from.c+d.dc;const md=Math.abs(nr-target.r)+Math.abs(nc-target.c)+Math.random()*0.3;if(md<bd){bd=md;best=d;}}\n  return best;\n}\n\n// state\nlet pac, dir, desiredDir, ghosts, dots, running, over, decisions, latency, cost, elapsed, ghostInfo, err;\nlet startAt, nextAskAt, ghostBusy, started, timer, elapsedTimer;\nfunction resetState(){\n  pac={...PAC_START}; dir=null; desiredDir=null;\n  ghosts=GHOST_STARTS.map(g=>({...g,dir:null}));\n  dots=initialDots(); running=false; over=null; decisions=0; latency=null; cost=0; elapsed=0;\n  ghostInfo=[null,null]; err=null; nextAskAt=0; ghostBusy=false; started=false;\n}\nresetState();\n\nconst board=document.getElementById("board");\nboard.style.gridTemplateColumns=`repeat(${COLS},1fr)`;\nboard.style.gridTemplateRows=`repeat(${ROWS},1fr)`;\nboard.style.aspectRatio=`${COLS}/${ROWS}`;\nconst cells=[];\nfor(let i=0;i<ROWS*COLS;i++){const d=document.createElement("div");board.appendChild(d);cells.push(d);}\n\nfunction render(){\n  for(let i=0;i<ROWS*COLS;i++){\n    const r=Math.floor(i/COLS),c=i%COLS,wall=WALLS[r][c];\n    const isPac=pac.r===r&&pac.c===c;\n    const gidx=ghosts.findIndex(g=>g.r===r&&g.c===c);\n    const hasDot=dots.has(kkey(r,c));\n    let cls="cell"+(wall?" wall":"");\n    let inner="";\n    if(!wall&&isPac)inner=\'<div class="pac"></div>\';\n    else if(!wall&&gidx>=0)inner=\'<div class="ghost g\'+(gidx%2)+\'"></div>\';\n    else if(!wall&&hasDot)inner=\'<div class="pdot"></div>\';\n    if(cells[i].className!==cls)cells[i].className=cls;\n    if(cells[i].innerHTML!==inner)cells[i].innerHTML=inner;\n  }\n  const eaten=TOTAL_DOTS-dots.size;\n  const st=over==="win"?"win":over==="caught"?"lose":running?"play":"";\n  const msg=over==="win"?"You win! 🏆 You outran Jev.":over==="caught"?"Jev caught you 👻":running?"Run! Jev\'s ghosts are hunting you":(decisions>0||elapsed>0?"Paused":"You drive Pac-Man — outrun Jev\'s ghosts");\n  const statusEl=document.getElementById("status");statusEl.textContent=msg;statusEl.className="status"+(st?" "+st:"");\n  document.getElementById("meta").innerHTML=`<span><b>${eaten}</b>/${TOTAL_DOTS} dots</span><span>· <b>${elapsed}</b>s</span><span>· <b>${decisions}</b> Jev calls</span><span>· <b>${latency!=null?latency:"—"}</b>ms</span><span>· <b>$${cost.toFixed(5)}</b></span>`;\n  const rows=[0,1].map(i=>{\n    const gi=ghostInfo[i];const pct=gi&&gi.prob!=null?Math.round(gi.prob*100):0;\n    return `<div class="hunt-row"><span class="hunt-arrow g${i}${gi?"":" idle"}">${gi?ARROW[gi.name]:"·"}</span>\n      <div class="hunt-main"><div class="hunt-top">\n        <span><span class="dot g${i}"></span>Ghost ${i+1}${gi?" → "+gi.name:" — waiting for Jev"}${gi&&gi.source==="fallback"?\'<span class="fb"> (fallback)</span>\':""}</span>\n        <b>${gi&&gi.prob!=null?gi.prob.toFixed(2):""}</b>\n      </div><div class="hunt-track"><div class="hunt-fill g${i}" style="width:${pct}%"></div></div></div></div>`;\n  }).join("");\n  document.getElementById("hunt-rows").innerHTML=rows;\n  const overEl=document.getElementById("over");\n  if(over){overEl.style.display="flex";overEl.className="over "+(over==="win"?"win":"lose");\n    overEl.innerHTML=`<b>${over==="win"?"You win! 🏆 You outran Jev.":"Jev caught you 👻"}</b><span>${eaten}/${TOTAL_DOTS} dots · survived ${elapsed}s · ${decisions} Jev decisions · $${cost.toFixed(5)}</span>`;\n  } else overEl.style.display="none";\n  const actions=document.getElementById("actions");\n  actions.innerHTML = (!running\n    ? `<button class="btn btn-primary" data-act="start">${over?"▶ Play again":(decisions>0||elapsed>0)?"▶ Resume":"▶ Start"}</button>`\n    : `<button class="btn" data-act="pause">⏸ Pause</button>`) + `<button class="btn" data-act="new">New game</button>`;\n  const e=document.getElementById("err");\n  if(err){e.style.display="block";e.textContent=err;}else e.style.display="none";\n}\n\nfunction stateText(){\n  let grid="";\n  for(let r=0;r<ROWS;r++){let row="";for(let c=0;c<COLS;c++){\n    if(WALLS[r][c]){row+="#";continue;}\n    if(pac.r===r&&pac.c===c){row+="P";continue;}\n    const gi=ghosts.findIndex(g=>g.r===r&&g.c===c);\n    if(gi===0)row+="1";else if(gi===1)row+="2";else if(dots.has(kkey(r,c)))row+=".";else row+=" ";\n  }grid+=row+"\\n";}\n  const heading=dir?dir.name:"still";\n  const gl=ghosts.map((g,i)=>{const dr=pac.r-g.r,dc=pac.c-g.c;const v=dr<0?`${-dr} up`:dr>0?`${dr} down`:"";const h=dc<0?`${-dc} left`:dc>0?`${dc} right`:"";const rel=[v,h].filter(Boolean).join(" and ")||"on the same tile";return `Ghost ${i+1} is at row ${g.r}, column ${g.c}; Pac-Man is ${rel} from it.`;});\n  return `Pac-Man maze, ${COLS} columns x ${ROWS} rows, 0-indexed. Legend: \'#\'=wall, \'.\'=dot, \'P\'=Pac-Man (the human you are hunting), \'1\'=ghost 1, \'2\'=ghost 2, \' \'=empty.\\n`+grid+`Pac-Man is at row ${pac.r}, column ${pac.c}, heading ${heading}.\\n`+gl.join("\\n")+"\\nYou control the ghosts. Corner and catch Pac-Man before he eats every dot.";\n}\nfunction ghostDirCriteria(g,d,target){const np={r:g.r+d.dr,c:g.c+d.dc};const md=Math.abs(np.r-target.r)+Math.abs(np.c-target.c);const onPac=np.r===target.r&&np.c===target.c;return `Head ${d.name} to row ${np.r}, column ${np.c} — ${onPac?"lands right on Pac-Man (a catch!)":`then ${md} tiles (Manhattan) from Pac-Man`}.`;}\n\nasync function askJev(){\n  ghostBusy=true;\n  const target=pac;const questions={};\n  for(let i=0;i<ghosts.length;i++){\n    const g=ghosts[i];const legal=legalDirs(g);const rev=g.dir?OPP[g.dir.name]:null;\n    let opts=legal.filter(d=>d.name!==rev);if(!opts.length)opts=legal;\n    if(opts.length<2)continue;\n    const criteria={};for(const d of opts)criteria[d.name]=ghostDirCriteria(g,d,target);\n    questions["ghost"+(i+1)]={type:"choice",instructions:i===0?"Ghost 1: pick the direction to best hunt Pac-Man. Cut him off; don\'t reverse into a wall.":"Ghost 2: pick the direction to hunt Pac-Man, ideally flanking from the other side.",criteria};\n  }\n  if(!Object.keys(questions).length){ghostBusy=false;return;}\n  const t=performance.now();\n  try{\n    const data=await jevDecide(stateText(),questions);\n    latency=Math.round(performance.now()-t);err=null;\n    cost+=((data&&data.usage&&data.usage.input_tokens||0)/1e6)*JEV_COST_PER_MTOK;\n    decisions++;\n    const info=ghostInfo.slice();\n    for(let i=0;i<ghosts.length;i++){\n      const ans=data&&data.answers&&data.answers["ghost"+(i+1)];if(!ans)continue;\n      const choice=ans.choice;const probs=ans.probabilities||ans.probs||{};\n      const prob=typeof probs[choice]==="number"?probs[choice]:typeof ans.confidence==="number"?ans.confidence:null;\n      const g=ghosts[i];\n      const picked=DIR_BY_NAME[choice]&&!isWall(g.r+DIR_BY_NAME[choice].dr,g.c+DIR_BY_NAME[choice].dc)?DIR_BY_NAME[choice]:null;\n      if(picked){g.dir=picked;info[i]={name:picked.name,prob,source:"jev"};}\n      else{const legal=legalDirs(g);const rev=g.dir?OPP[g.dir.name]:null;const nonRev=legal.filter(d=>d.name!==rev);const fb=greedyToward(g,nonRev.length?nonRev:legal,pac);if(fb){g.dir=fb;info[i]={name:fb.name,prob:null,source:"fallback"};}}\n    }\n    ghostInfo=info;ghostBusy=false;\n  }catch(e){\n    if(e.code===429){err="Jev is rate-limited — ghosts hold their course…";nextAskAt=performance.now()+RATE_WAIT_MS;ghostBusy=false;return;}\n    err=String(e.message||e);\n    const info=ghostInfo.slice();\n    for(let i=0;i<ghosts.length;i++){const g=ghosts[i];const legal=legalDirs(g);const rev=g.dir?OPP[g.dir.name]:null;const nonRev=legal.filter(d=>d.name!==rev);const fb=greedyToward(g,nonRev.length?nonRev:legal,pac);if(fb){g.dir=fb;info[i]={name:fb.name,prob:null,source:"fallback"};}}\n    ghostInfo=info;nextAskAt=performance.now()+THROTTLE_MS;ghostBusy=false;\n  }\n}\nfunction maybeAskJev(){if(ghostBusy)return;if(performance.now()<nextAskAt)return;nextAskAt=performance.now()+THROTTLE_MS;askJev();}\nfunction stepGhosts(){\n  const target=pac;\n  ghosts=ghosts.map(g=>{\n    const legal=legalDirs(g);if(!legal.length)return g;\n    const canStraight=g.dir&&!isWall(g.r+g.dir.dr,g.c+g.dir.dc);\n    let move;\n    if(canStraight)move=g.dir;\n    else{const rev=g.dir?OPP[g.dir.name]:null;const nonRev=legal.filter(d=>d.name!==rev);move=greedyToward(g,nonRev.length?nonRev:legal,target)||legal[0];}\n    return {r:g.r+move.dr,c:g.c+move.dc,dir:move};\n  });\n}\nfunction finish(kind){running=false;over=kind;stopLoop();render();}\nfunction tick(){\n  if(!running)return;\n  const from=pac,cur=dir,des=desiredDir;\n  let move=null;\n  if(des&&!isWall(from.r+des.dr,from.c+des.dc))move=des;\n  else if(cur&&!isWall(from.r+cur.dr,from.c+cur.dc))move=cur;\n  if(move){\n    pac={r:from.r+move.dr,c:from.c+move.dc};dir=move;\n    if(dots.has(kkey(pac.r,pac.c))){dots=new Set(dots);dots.delete(kkey(pac.r,pac.c));}\n    if(dots.size===0){finish("win");return;}\n    if(ghosts.some(g=>g.r===pac.r&&g.c===pac.c)){finish("caught");return;}\n  }\n  maybeAskJev();\n  if(!ghostBusy)stepGhosts();\n  if(ghosts.some(g=>g.r===pac.r&&g.c===pac.c)){finish("caught");return;}\n  render();\n}\nfunction startLoop(){stopLoop();timer=setInterval(tick,STEP_MS);startAt=performance.now()-elapsed*1000;elapsedTimer=setInterval(()=>{const s=Math.floor((performance.now()-startAt)/1000);if(s!==elapsed){elapsed=s;render();}},250);}\nfunction stopLoop(){clearInterval(timer);clearInterval(elapsedTimer);}\nfunction start(){if(over)resetState();err=null;running=true;started=true;startLoop();board.focus();render();}\nfunction pause(){running=false;stopLoop();render();}\nfunction newGame(){stopLoop();resetState();render();}\n\n// input\nwindow.addEventListener("keydown",e=>{if(!running||over)return;const n=KEY_TO_DIR[e.key];if(!n)return;e.preventDefault();desiredDir=DIR_BY_NAME[n];});\nboard.addEventListener("keydown",e=>{if(over)return;const n=KEY_TO_DIR[e.key];if(!n)return;e.preventDefault();desiredDir=DIR_BY_NAME[n];if(!running)start();});\nboard.addEventListener("click",()=>{if(!running&&!over)start();});\ndocument.querySelectorAll(".dpad button").forEach(b=>b.addEventListener("click",()=>{if(over)return;desiredDir=DIR_BY_NAME[b.dataset.dir];if(!running)start();}));\ndocument.getElementById("actions").addEventListener("click",e=>{const a=e.target.dataset.act;if(a==="start")start();else if(a==="pause")pause();else if(a==="new")newGame();});\ndocument.getElementById("over").addEventListener("click",()=>{});\n\nrender();\n</script>\n</body>\n</html>\n', 'community/LICENSE': 'MIT License\n\nCopyright (c) 2026 codaaiteam\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'}
for name, content in BUNDLED_FILES.items():
    target = LAB_DIR / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
sys.path.insert(0, str(LAB_DIR))
from cloud_runtime import CloudRuntime, MODEL_RUN
from pacman_lab import *
from api_client import call, distribution
SOURCE_HTML = (LAB_DIR / 'community/index.html').read_text()
MAZE = maze_from_html(SOURCE_HTML)
runtime = CloudRuntime(LAB_DIR)
manifest = make_data(SOURCE_HTML, LAB_DIR / 'data')
GAME = notebook_game(SOURCE_HTML, (LAB_DIR / 'games/player-controller.js').read_text(), (LAB_DIR / 'community/LICENSE').read_text())
print('Prepared player controller and disjoint synthetic snapshots.')

In [ ]:
runtime.setup()
audit = runtime.prepare_training()
print('Training suite records:', audit['suite_records'])
print('Trainable parameters:', sum(audit['trainable_parameters'].values()))

## Prework: initial decision training from the base

This cell uses the unchanged upstream `kev.train` module and the selected published experiment (`experiments/q35-08b.json`, seed 2). It uses all **12,576 decision-v7 training records**, two epochs, batch 8, accumulation 1, LoRA rank 16 / alpha 32, a 256-dimensional pointer head, learning rate 1e-4, OneCycleLR, AdamW and BF16 autocast over FP32 weights. It preserves option shuffling, none-of-the-above/distractor augmentation and 25% none minimal pairs. It passes **no `--init_from`** and never uses development or test data for training.

The published H100 initial run took about 20 minutes. We have not timed this Colab GPU. Run it before class and keep its checkpoint. The helper caps the attempt at 90 minutes. Use a new directory for a new attempt. A supplied instructor checkpoint is a fallback and must be labelled as such.

The next cell defaults to full initial training. To restore saved prework after a disconnect, upload its ZIP through Colab's Files sidebar and set `RESTORE_ARCHIVE` to its path. Declare whether the checkpoint is yours or the instructor's. The following cell saves a backup ZIP.

In [ ]:
INITIAL = LAB_DIR / 'checkpoints/decision-v7-initial'
print('Published command:', runtime.pretraining_command(INITIAL))
RESTORE_ARCHIVE = None  # Example: '/content/pacman-initial-checkpoint.zip'
RESTORED_CHECKPOINT_OWNER = 'learner'  # Set to 'instructor' for a supplied fallback
import zipfile
if RESTORE_ARCHIVE is None:
    initial_checkpoint = runtime.pretrain(INITIAL, steps=0)
    initial_origin = 'learner trained in this session'
else:
    INITIAL.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(RESTORE_ARCHIVE) as saved:
        assert all((INITIAL / member.filename).resolve().is_relative_to(INITIAL.resolve()) for member in saved.infolist())
        saved.extractall(INITIAL)
    initial_checkpoint = INITIAL
    initial_origin = RESTORED_CHECKPOINT_OWNER + ' restored checkpoint'
initial_config = json.loads((INITIAL / 'training_config.json').read_text())
initial_metrics = json.loads((INITIAL / 'training_metrics.json').read_text())
assert initial_config['init_source'] is None, 'Initial training must start from the base'
published = json.loads((runtime.repo / 'experiments/q35-08b.json').read_text())[2]
assert all(initial_config['args'][key] == value for key,value in published.items()), 'Restore the matching published base-stage checkpoint'
assert initial_metrics['optimizer_steps'] > 0
assert initial_metrics['records_seen'] == initial_metrics['requested_records'], 'Complete both epochs'
models = runtime.start(INITIAL)
print(json.dumps(models, indent=2))

In [ ]:
PREWORK_ARCHIVE = LAB_DIR.parent / 'pacman-initial-checkpoint.zip'
with zipfile.ZipFile(PREWORK_ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as saved:
    for file in INITIAL.rglob('*'):
        if file.is_file():
            saved.write(file, file.relative_to(INITIAL))
try:
    from google.colab import files
except ImportError:
    print('Save your initial checkpoint:', PREWORK_ARCHIVE)
else:
    files.download(str(PREWORK_ARCHIVE))

## CP0: play, then give Kev the controls (0–10 minutes)

Try **Human** mode with arrows/WASD. Restart, select **Kev**, and watch its choices. The ghosts move after every second player turn. The whole simulation waits for each model answer; wall-clock survival is therefore not a fair skill metric. Pause the board before running training cells.

Colab supplies the notebook callback below. On Kaggle, use the structured decision cell and Python rollouts instead. API errors pause visibly; the game does not replace failed player decisions with a hidden rules controller.

In [ ]:
from IPython.display import display, HTML, JSON
bridge = NotebookBridge(LAB_DIR / 'results/player-trace.jsonl')
try:
    from google.colab import output
except ImportError:
    print('Kaggle/local: continue with the decision and rollout cells below.')
else:
    output.register_callback('pacman.decide', lambda state: JSON(bridge.decide(state)))
    display(HTML(GAME))

## CP1: inspect initial training and a player decision (10–20 minutes)

Inspect `initial_config`, `initial_metrics` and `trainable-parameters.json`. Find the fresh pointer head, trainable LoRA parameters and fixed original base matrices. Explain why gradients still travel through the encoder. Initial training learns general typed decisions; it has not yet seen Pac-Man labels.

Predict the move first. Identify the player, two ghosts, remaining dots and legal options. Explain why a direction through a wall never appears. The game consumes `answers.move.choice`; inspect probabilities by direction name. Confidence is not the probability of clearing the maze.

In [ ]:
state = initial_state(MAZE)
request = body(state)
print('Player:', state['player'], '\nGhosts:', state['ghosts'])
print('Options:', request['questions']['move']['criteria'])
response, elapsed = call('/v1/systemone', request)
distribution(response['answers']['move'], request['questions']['move']['criteria'])
print(json.dumps(response['answers'], indent=2), '\nHTTP ms:', round(elapsed))

## CP2: inspect and edit labels (20–30 minutes)

Starter data has **64 training, 16 development, 16 evaluation** snapshots. Labels come from a stated heuristic: avoid immediate capture, approach a dot by maze distance, prefer distance from ghosts, then use a fixed tie order. These are synthetic labels, not recorded human expertise.

Inspect three training boards and enter your own legal move labels in `EDITS` before checking the teacher. Keep all edits in the training partition. Fix the training file and one candidate before opening evaluation data. Positions are disjoint across splits, but every split uses the same maze.

In [ ]:
training = [json.loads(line) for line in (LAB_DIR / 'data/pacman-train.jsonl').read_text().splitlines()]
for row in training[:3]:
    print(row['_meta']['id'], json.dumps(row['state'], indent=2))
    print('Legal:', list(row['questions']['move']['criteria']))
EDITS = {}  # Example after inspecting a board: {'board-000': 'left'}
assert set(EDITS).issubset({r['_meta']['id'] for r in training}), 'Unknown training board ID'
for row in training:
    if row['_meta']['id'] in EDITS:
        label = EDITS[row['_meta']['id']]
        assert label in row['questions']['move']['criteria'], 'Choose a legal move'
        row['questions']['move'].update(label=label, src='learner_annotation')
        row['_meta']['label_source'] = 'learner annotation'
training_file = LAB_DIR / 'data/pacman-train-reviewed.jsonl'
training_file.write_text(''.join(json.dumps(row) + '\n' for row in training))
print('Teacher labels for inspected boards:', [(r['_meta']['id'], r['questions']['move']['label']) for r in training[:3]])
print('Split sizes:', manifest['counts'])
before_dev = evaluate(LAB_DIR / 'data/pacman-development.jsonl')
print('Development accuracy:', before_dev['accuracy'])

## CP3: fine-tune Pac-Man decisions (30–60 minutes)

30-minute block: 5 minutes inspect a labelled request and the loss; up to 20 minutes fine-tune for **two complete epochs**; 5 minutes inspect and save the checkpoint. This is supervised imitation. Kev updates its LoRA adapter and pointer head while the original base matrices remain frozen; the adapter changes the encoder's effective features.

Warm-start from **your initial decision-v7 checkpoint**, using Kev's documented custom-data settings: learning rate 2e-5, batch 1, accumulation 8, BF16 autocast and gradient checkpointing. With 64 accepted rows, this gives 16 optimizer steps. The Pac-Man adaptation disables none/distractor insertion because the only valid outputs are legal player moves. Option shuffling remains active. This is a documented task-specific departure from the generic initial recipe.

The helper reads architecture from the checkpoint and stops inference to free GPU memory. Use a new checkpoint directory. A timed GPU preflight is still required; successful execution or improved play has not been established by this draft.

In [ ]:
print(json.dumps(training[0], indent=2))
CHECKPOINT = LAB_DIR / 'checkpoints/pacman-finetuned'
checkpoint = runtime.finetune(training_file, CHECKPOINT, init_from=INITIAL, steps=0)
metrics = json.loads((checkpoint / 'training_metrics.json').read_text())
print(metrics)
assert metrics['optimizer_steps'] > 0, 'Training must update the model'
assert metrics['records_seen'] == metrics['requested_records'], 'Complete both epochs'

## CP4: before/after on the same decisions (60–72 minutes)

The candidate is now fixed. Score both models on the same 16 evaluation snapshots and save predictions by ID. Report accuracy against the teacher and immediate captures after the selected moves. Sixteen boards on one maze do not establish general game skill. Fine-tuning may leave answers unchanged or make them worse.

In [ ]:
runtime.start(INITIAL)
before = evaluate(LAB_DIR / 'data/pacman-evaluation.jsonl')
before_run = rollout(MAZE, max_turns=24)
runtime.start(checkpoint)
after = evaluate(LAB_DIR / 'data/pacman-evaluation.jsonl')
after_run = rollout(MAZE, max_turns=24)
comparison = {'base_model': audit['base'], 'base_revision': audit['base_revision'], 'initial_checkpoint': str(INITIAL), 'initial_origin': initial_origin, 'fine_tuned_checkpoint': str(checkpoint), 'before': before, 'after': after, 'rollouts': {'initial': before_run, 'fine_tuned': after_run}, 'initial_training': initial_metrics, 'pacman_training': metrics, 'runtime': runtime.gpu}
(LAB_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2))
for name, result in [('initial', before), ('fine_tuned', after)]:
    print(name, 'accuracy', result['accuracy'], 'immediate captures', result['caught_next_turn'])

## CP4 continued: watch Pac-Man play (72–80 minutes)

Repeat the browser cell, choose **Kev**, and start a new game with the fine-tuned checkpoint active. Both Python rollouts above use the same starting board, ghost code and 24-turn cap. Compare dots and turns; these two trajectories are an illustration, not a win-rate estimate.

The starter teacher is also a useful rules baseline. It is deliberately simple and can get stuck. The model learns from structured state; there is no screenshot encoder or frame-by-frame RL training in this exercise.

In [ ]:
for name, episode in [('initial', before_run), ('fine_tuned', after_run)]:
    print(name, {k:episode[k] for k in ['turns', 'dots_collected', 'outcome']})
# Optional rules baseline using the same game mechanics:
def rule_predict(request):
    chosen = teacher(request['state'])
    keys = request['questions']['move']['criteria']
    return {'answers': {'move': {'choice':chosen, 'probabilities':{k:float(k==chosen) for k in keys}}}}
rule_run = rollout(MAZE, predict=rule_predict, max_turns=24)
print('rules', {k:rule_run[k] for k in ['turns','dots_collected','outcome']})

## CP5: explain and submit (80–90 minutes)

Explain one changed move and one confident mistake. Trace `kev/api.py:to_record`, `kev/model.py:encode` / `PointerHead`, and `kev/train.py`. The lecture's CLM uses separate state/action representations and InfoNCE; Kev scores option-token representations with a pointer head and supervised cross-entropy. They are related decision systems with different training architectures.

Submit the executed notebook, reviewed training JSONL, `comparison.json`, both small adapter/head checkpoints, both training configurations/metrics and `runtime-preflight.json`. Record the Colab compute units consumed and elapsed GPU time from your session. Save outputs before the temporary runtime disconnects, then stop the server. On Colab, run the download cell.

In [ ]:
runtime.stop()
archive = str(LAB_DIR.parent / 'pacman-lab-submission.zip')
# Export small adapters/heads and results, without foundation weights or packages.
import zipfile
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as out:
    for file in [LAB_DIR/'comparison.json', training_file, LAB_DIR/'runtime-preflight.json', LAB_DIR/'trainable-parameters.json']:
        out.write(file, file.relative_to(LAB_DIR))
    for folder in [INITIAL, checkpoint]:
        for file in folder.rglob('*'):
            if file.is_file():
                out.write(file, file.relative_to(LAB_DIR))
try:
    from google.colab import files
except ImportError:
    print('Download:', archive)
else:
    files.download(archive)